In [ ]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
sellout_test = pd.read_csv(r"../data/Sellout_Prediction_Support_Data.csv")
ihs_test = pd.read_excel(r"../data/IHS Test Data.xlsx")


sellout_test['SKU_ID'] = (
    sellout_test['brand'].astype(str) + "_" + 
    sellout_test['sub_brand'].astype(str) + "_" + 
    sellout_test['package'].astype(str) + "_" + 
    sellout_test['package_type'].astype(str) + "_" + 
    sellout_test['capacity_number'].astype(str)
)

sellout_test['date'] = pd.to_datetime(sellout_test['date'])
sellout_test['year_month'] = sellout_test['date'].dt.to_period('M')
ihs_test['year_month'] = ihs_test['Date'].dt.to_period('M')


test_df = pd.merge(sellout_test, ihs_test, on = 'year_month', how='outer')


full_df_test = test_df.copy()
full_df_test['log_price'] = np.log(full_df_test['avg_price_per_liter'])
full_df_test['log_nd'] = np.log1p(full_df_test['numeric_distribution_stores_handling'])
full_df_test['log_inv'] = np.log1p(full_df_test['inventory_hectoliters'])
full_df_test['log_wd'] = np.log1p(full_df_test['weighted_distribution_tdp_reach'])


ihs_vars = [
    'consumer price index, core, usd',
    'gdp per capita, real, harmonized',
    'unemployment rate', 'wholesale price index, usd', 'primary income, bop, usd', 'fixed investment, real, harmonized',
    'domestic demand, real, harmonized', 'interest rate, short-term, real'
]



from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
#full_df_test[ihs_vars] = scaler.fit_transform(full_df_test[ihs_vars])


full_df_test['year_month'] = full_df_test['year_month'].dt.to_timestamp()




sellout = pd.read_excel(r"../data/Training Data Sellout.xlsx")
ihs = pd.read_excel(r"../data/IHS Data.xlsx")

sellout['SKU_ID'] = (
    sellout['brand'].astype(str) + "_" + 
    sellout['sub_brand'].astype(str) + "_" + 
    sellout['package'].astype(str) + "_" + 
    sellout['package_type'].astype(str) + "_" + 
    sellout['capacity_number'].astype(str)
)

sellout['date'] = pd.to_datetime(sellout['date'])
sellout['year_month'] = sellout['date'].dt.to_period('M')
ihs['year_month'] = ihs['Date'].dt.to_period('M')


train_df = pd.merge(sellout, ihs, on = 'year_month', how='outer')


full_df = train_df.copy()
full_df['log_volume'] = np.log1p(full_df['sales_hectoliters'])
full_df['log_price'] = np.log(full_df['avg_price_per_liter'])
full_df['log_nd'] = np.log1p(full_df['numeric_distribution_stores_handling'])
full_df['log_inv'] = np.log1p(full_df['inventory_hectoliters'])
full_df['log_wd'] = np.log1p(full_df['weighted_distribution_tdp_reach'])


ihs_vars = [
    'consumer price index, core, usd',
    'gdp per capita, real, harmonized',
    'unemployment rate', 'wholesale price index, usd', 'primary income, bop, usd', 'fixed investment, real, harmonized',
    'domestic demand, real, harmonized', 'interest rate, short-term, real'
]



from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
#full_df_test[ihs_vars] = scaler.fit_transform(full_df_test[ihs_vars])


full_df['year_month'] = full_df['year_month'].dt.to_timestamp()


In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

In [ ]:
# TRAIN ALL DATES LOG

#features = ['log_price','log_nd','log_inv','log_wd'] + ihs_vars + ['year','month','SKU_ID']
features = ['avg_price_per_liter','numeric_distribution_stores_handling','inventory_hectoliters','weighted_distribution_tdp_reach'] + ihs_vars + ['year','month','SKU_ID']
#target = 'log_volume'
target = 'log_volume'

train_df = full_df[full_df['year_month'] < "2024-01-01"]
test_df  = full_df[full_df['year_month'] >= "2024-01-01"]

for df in [train_df, test_df]:
    df['SKU_ID'] = df['SKU_ID'].astype('category')
    df['year'] = df['year_month'].dt.year
    df['month'] = df['year_month'].dt.month

X_train, y_train = train_df[features], train_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

X_train = X_train.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_test  = X_test.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

feature_names = list(X_train.columns)

constraints = [-1] + [0] * (len(features) - 1)

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=['SKU_ID'])
valid_set = lgb.Dataset(X_test, label=y_test, categorical_feature=['SKU_ID'])


# model_ad = lgb.LGBMRegressor(
#     objective="regression",
#     n_estimators=1000,
#     learning_rate=0.05,
#     num_leaves=31,
#     max_depth=-1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )

model_ad = lgb.LGBMRegressor(
    objective="regression",
    metric="rmse",
    boosting_type="gbdt",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    monotone_constraints=constraints,
    random_state=42
)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(100)
]

model_ad.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_metric="rmse",
    callbacks=callbacks,
    categorical_feature=["SKU_ID"],
    feature_name=feature_names
)

# Predictions
y_pred_train = model_ad.predict(X_train, num_iteration=model_ad.best_iteration_)
print("RMSE Train:", mean_squared_error(y_train, y_pred_train, squared=False))
print("R² Train:", r2_score(y_train, y_pred_train))
print('\n')
# Predictions
y_pred = model_ad.predict(X_test, num_iteration=model_ad.best_iteration_)
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("R²:", r2_score(y_test, y_pred))

In [ ]:
full_df_test_sub = full_df_test[full_df_test['SKU_ID'].isin(full_df['SKU_ID'].unique())]
pred_df  = full_df_test_sub.copy()
pred_df['SKU_ID'] = pred_df['SKU_ID'].astype('category')
pred_df['year'] = pred_df['year_month'].dt.year
pred_df['month'] = pred_df['year_month'].dt.month

pred_df = pred_df.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_future = pred_df[X_train.columns]

pred_df["log_volume_pred"] = model_ad.predict(X_future, num_iteration=model_ad.best_iteration_)

pred_df["pred_volume"] = np.expm1(pred_df["log_volume_pred"])

# pred_df["volume_pred"] = model_ad.predict(X_future, num_iteration=model_ad.best_iteration_)

pred_df.head(10)

In [ ]:
#pred_df.to_excel('lightgbm test data predictions LOG.xlsx',index=False)

In [ ]:
# VERSION WITH SKU ID NOT CATEGORICAL

categorical_feats = ["brand", "sub_brand", "package", "package_type", "capacity_number", "year", "month"]
#numeric_feats = ["log_price", "log_nd", "log_inv", "log_wd"] + ihs_vars
numeric_feats = ['avg_price_per_liter','numeric_distribution_stores_handling','inventory_hectoliters','weighted_distribution_tdp_reach'] + ihs_vars
#target = 'log_volume'
target = 'sales_hectoliters'

training_full = full_df.copy()
pred_full = full_df_test.copy()

for df in [training_full, pred_full]:
    df['year'] = df['year_month'].dt.year
    df['month'] = df['year_month'].dt.month

for col in categorical_feats:
    training_full[col] = training_full[col].astype("category")
    pred_full[col]  = pred_full[col].astype("category")

train_df = training_full[training_full['year_month'] < "2024-10-01"]
test_df  = training_full[training_full['year_month'] >= "2024-10-01"]


X_train, y_train = train_df[categorical_feats + numeric_feats], train_df[target]
X_val,  y_val  = test_df[categorical_feats + numeric_feats],  test_df[target]

X_train = X_train.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_val  = X_val.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

feature_names = list(X_train.columns)


train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_feats)
valid_set = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_feats)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1
}

model = lgb.train(
    params,
    train_set,
    valid_sets=[train_set, valid_set],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)


# Predictions
y_pred_train = model.predict(X_train)
print("RMSE Train:", mean_squared_error(y_train, y_pred_train, squared=False))
print("R² Train:", r2_score(y_train, y_pred_train))
print('\n')
# Predictions
y_pred = model.predict(X_val)
print("RMSE:", mean_squared_error(y_val, y_pred, squared=False))
print("R²:", r2_score(y_val, y_pred))

In [ ]:
X_future = pred_full[categorical_feats + numeric_feats]
X_future = X_future.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

# # Predict log volumes
# pred_full["log_volume_pred"] = model.predict(X_future)

# # Back to original scale
# pred_full["pred_volume"] = np.expm1(pred_full["log_volume_pred"])


pred_full["volume_pred"] = model.predict(X_future)

pred_full[["year_month", "brand", "sub_brand", "package", "package_type", "capacity_number", "volume_pred"]].head()


In [ ]:
#pred_full.to_excel('lightgbm test data predictions SKU SEPARATE.xlsx',index=False)

In [ ]:
# TRAIN ALL DATES SKU CATEGORICAL

#features = ['log_price','log_nd','log_inv','log_wd'] + ihs_vars + ['year','month','SKU_ID']
features = ['avg_price_per_liter','numeric_distribution_stores_handling','inventory_hectoliters','weighted_distribution_tdp_reach'] + ihs_vars + ['year','month','SKU_ID']
#target = 'log_volume'
target = 'sales_hectoliters'

# Train/test split
train_df = full_df[full_df['year_month'] < "2024-01-01"]
test_df  = full_df[full_df['year_month'] >= "2024-01-01"]

for df in [train_df, test_df]:
    df['SKU_ID'] = df['SKU_ID'].astype('category')
    df['year'] = df['year_month'].dt.year
    df['month'] = df['year_month'].dt.month

X_train, y_train = train_df[features], train_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

X_train = X_train.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_test  = X_test.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

feature_names = list(X_train.columns)

constraints = [-1] + [0] * (len(features) - 1)

train_set = lgb.Dataset(X_train, label=y_train, categorical_feature=['SKU_ID'])
valid_set = lgb.Dataset(X_test, label=y_test, categorical_feature=['SKU_ID'])


# model_ad = lgb.LGBMRegressor(
#     objective="regression",
#     n_estimators=1000,
#     learning_rate=0.05,
#     num_leaves=31,
#     max_depth=-1,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     random_state=42
# )

model_ad = lgb.LGBMRegressor(
    objective="regression",
    metric="rmse",
    boosting_type="gbdt",
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    monotone_constraints=constraints,
    random_state=42
)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(100)
]

model_ad.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    eval_metric="rmse",
    callbacks=callbacks,
    categorical_feature=["SKU_ID"],
    feature_name=feature_names
)

# Predictions
y_pred_train = model_ad.predict(X_train, num_iteration=model_ad.best_iteration_)
print("RMSE Train:", mean_squared_error(y_train, y_pred_train, squared=False))
print("R² Train:", r2_score(y_train, y_pred_train))
print('\n')
# Predictions
y_pred = model_ad.predict(X_test, num_iteration=model_ad.best_iteration_)
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("R²:", r2_score(y_test, y_pred))

In [ ]:
full_df_test_sub = full_df_test[full_df_test['SKU_ID'].isin(full_df['SKU_ID'].unique())]
pred_df  = full_df_test_sub.copy()
pred_df['SKU_ID'] = pred_df['SKU_ID'].astype('category')
pred_df['year'] = pred_df['year_month'].dt.year
pred_df['month'] = pred_df['year_month'].dt.month

pred_df = pred_df.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_future = pred_df[X_train.columns]

# # Predict log volumes
# pred_df["log_volume_pred"] = model_ad.predict(X_future)

# # Convert back to original scale
# pred_df["pred_volume"] = np.expm1(pred_df["log_volume_pred"])

# Predict log volumes
pred_df["volume_pred"] = model_ad.predict(X_future, num_iteration=model_ad.best_iteration_)

pred_df.head(10)

In [ ]:
import optuna

def optuna_tuning(trial):
    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "n_estimators": 2000,
        "learning_rate": trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 20),
        "subsample": trial.suggest_uniform("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_uniform("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_loguniform("reg_alpha", 1e-8, 10.0),
        "reg_lambda": trial.suggest_loguniform("reg_lambda", 1e-8, 10.0),
        "random_state": 42
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(100)
    ]


    model_opt = lgb.LGBMRegressor(**params)
    model_opt.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        eval_metric="rmse",
        callbacks = callbacks,
        #verbose=-1,
        categorical_feature=["SKU_ID"],
        feature_name=feature_names
    )
    
    y_pred = model_opt.predict(X_test)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    return rmse


study = optuna.create_study(direction="minimize")
study.optimize(optuna_tuning, n_trials=500)

print("Best params:", study.best_params)
print("Best RMSE:", study.best_value)

In [ ]:
train_df = full_df[full_df['year_month'] < "2024-11-01"]
test_df  = full_df[full_df['year_month'] >= "2024-11-01"]

for df in [train_df, test_df]:
    df['SKU_ID'] = df['SKU_ID'].astype('category')
    df['year'] = df['year_month'].dt.year
    df['month'] = df['year_month'].dt.month

X_train, y_train = train_df[features], train_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

X_train = X_train.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_test  = X_test.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

feature_names = list(X_train.columns)

best_params = study.best_params
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 2000,
    "random_state": 42
})

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(100)
]

model_opt_best = lgb.LGBMRegressor(**best_params)

model_opt_best.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)], 
    eval_metric="rmse",
    callbacks = callbacks,
    #verbose=-1,
    categorical_feature=["SKU_ID"],
    feature_name=feature_names
)

# Predictions
y_pred_train = model_opt_best.predict(X_train, num_iteration=model_opt_best.best_iteration_)
print("RMSE Train:", mean_squared_error(y_train, y_pred_train, squared=False))
print("R² Train:", r2_score(y_train, y_pred_train))
print('\n')


In [ ]:
full_df_test_sub = full_df_test[full_df_test['SKU_ID'].isin(full_df['SKU_ID'].unique())]
pred_df  = full_df_test_sub.copy()
pred_df['SKU_ID'] = pred_df['SKU_ID'].astype('category')
pred_df['year'] = pred_df['year_month'].dt.year
pred_df['month'] = pred_df['year_month'].dt.month

pred_df = pred_df.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_future = pred_df[X_train.columns]

# # Predict log volumes
# pred_df["log_volume_pred"] = model_ad.predict(X_future)

# # Convert back to original scale
# pred_df["pred_volume"] = np.expm1(pred_df["log_volume_pred"])

# Predict log volumes
pred_df["volume_pred"] = model_opt_best.predict(X_future, num_iteration=model_opt_best.best_iteration_)

pred_df.head(10)

In [ ]:
import pandas as pd
import numpy as np

def simulation(model, df, price_col="avg_price_per_liter", 
                               target_col="sales_hectoliters", sku_col="SKU_ID"):
    
    df_up = df.copy()
    df_up[price_col] *= 1.02   # +1% price

    q_pred = model.predict(df)
    q_up = model.predict(df_up)
    q_pred = np.where(q_pred <= 0, 1e-6, q_pred)
    difference = ((q_up - q_pred) / q_pred) / 0.02

    detailed_df = df.copy()
    detailed_df["Prediction Upper"] = q_up
    detailed_df["Prediction"] = q_pred
    detailed_df["Difference"] = difference
    detailed_df["Price"] = df[price_col]

    per_sku_difference = (
        detailed_df.groupby(sku_col)
        .apply(lambda g: pd.Series({
            "Mean_Difference": g["Difference"].mean(),
            "Median_Difference": g["Difference"].median(),
            "VolWeighted_Difference": np.average(
                g["Difference"], weights=g["Prediction"]
            )
        }))
        .reset_index()
    )

    return per_sku_difference, detailed_df


In [ ]:
per_sku_difference, detailed_df = simulation(
    model_ad, 
    df=X_train,  # or train_df if you want elasticities in-sample
    price_col="avg_price_per_liter",
    target_col="sales_hectoliters",
    sku_col="SKU_ID"
)

In [ ]:
import numpy as np
import pandas as pd

deltas = [0.01, 0.05, 0.10]
y_pred_base = model_ad.predict(X_train, num_iteration=model_ad.best_iteration_)
perturbed_preds = {}

for d in deltas:
    X_perturbed = X_train.copy()
    X_perturbed["avg_price_per_liter"] = X_train["avg_price_per_liter"] * (1 + d)
    y_pred_perturbed = model_ad.predict(X_perturbed, num_iteration=model_ad.best_iteration_)
    perturbed_preds[d] = y_pred_perturbed

differences_all = []
for d in deltas:
    difference = ((perturbed_preds[d] - y_pred_base) / y_pred_base) / d
    differences_all.append(difference)

difference_avg = np.mean(differences_all, axis=0)

detailed_df = X_train.copy()
detailed_df["actual_sales"] = y_train.values
detailed_df["pred_sales"] = y_pred_base
detailed_df["difference"] = difference_avg

for d in deltas:
    colname = f"pred_sales_plus{int(d*100)}pct"
    detailed_df[colname] = perturbed_preds[d]

sku_difference = (
    detailed_df
    .groupby("SKU_ID")
    .apply(lambda g: np.average(g["difference"], weights=g["pred_sales"]))
    .reset_index(name="avg_price_difference")
)


In [ ]:
ce_df = full_df.copy()

ce_df["brand_avg_price"] = ce_df.groupby(["brand", "year_month"])["avg_price_per_liter"].transform("mean")
ce_df["subbrand_avg_price"] = ce_df.groupby(["sub_brand", "year_month"])["avg_price_per_liter"].transform("mean")
ce_df["package_avg_price"] = ce_df.groupby(["package_type", "year_month"])["avg_price_per_liter"].transform("mean")


In [ ]:
import numpy as np
import pandas as pd

def make_combo_keys(df, combos=None):
    if combos is None:
        combos = [
            ("brand", "sub_brand"),
            ("brand", "package"),
            ("sub_brand", "package"),
            ("brand", "sub_brand", "package")
        ]
    for combo in combos:
        key = "_".join(combo)
        df[key] = df[list(combo)].astype(str).agg("_".join, axis=1)
    return df

def add_loo_competitor_prices(df, group_cols, price_col="avg_price_per_liter", date_col="date", min_group_size=2):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    grp = df.groupby(group_cols + [date_col])[price_col]
    group_sum = grp.transform("sum")
    group_count = grp.transform("count")
    out_col = "_".join(group_cols) + "_price_excl_sku"
    df[out_col] = (group_sum - df[price_col]) / (group_count - 1).clip(lower=1)
    df.loc[group_count <= 1, out_col] = np.nan
    if min_group_size > 2:
        df.loc[group_count < min_group_size, out_col] = np.nan
    return df

def compute_agg_elasticities_multidelta(model, X, agg_cols, deltas=[0.01,0.05,0.10]):
    y_base = model.predict(X)
    results = []
    for comp_col in agg_cols:
        if comp_col not in X.columns:
            continue
        elasticity_list = []
        for d in deltas:
            Xp = X.copy()
            Xp[comp_col] = X[comp_col] * (1.0 + d)
            y_pert = model.predict(Xp)
            denom = np.where(y_base <= 0, 1e-8, y_base)
            elasticity = ((y_pert - y_base) / denom) / d
            elasticity_list.append(elasticity)
        elasticity_mean = np.mean(elasticity_list, axis=0)
        tmp = pd.DataFrame({
            "SKU_ID": X["SKU_ID"].values,
            "feature": comp_col,
            "elasticity": elasticity_mean,
            "baseline_sales": y_base
        })
        results.append(tmp)
    if len(results) == 0:
        return pd.DataFrame(columns=["SKU_ID","feature","elasticity","baseline_sales"])
    return pd.concat(results, ignore_index=True)


ce_df2 = full_df.copy()
#full_df['date'] = pd.to_datetime(full_df['date'])

combos = [
    ("brand","sub_brand"),
    ("brand","package"),
    ("sub_brand","package"),
    ("brand","sub_brand","package")
]
ce_df2 = make_combo_keys(ce_df2, combos=combos)

ce_df2 = add_loo_competitor_prices(ce_df2, "brand", price_col="avg_price_per_liter", date_col="year_month", min_group_size=2)
ce_df2 = add_loo_competitor_prices(ce_df2, "sub_brand", price_col="avg_price_per_liter", date_col="year_month", min_group_size=2)
ce_df2 = add_loo_competitor_prices(ce_df2, "package", price_col="avg_price_per_liter", date_col="year_month", min_group_size=2)

for combo in combos:
    key = "_".join(combo)
    ce_df2 = add_loo_competitor_prices(ce_df2, key, price_col="avg_price_per_liter", date_col="year_month", min_group_size=3)

In [ ]:
features = [
    "avg_price_per_liter", "brand_avg_price", "subbrand_avg_price", "package_avg_price",
    "numeric_distribution_stores_handling", "inventory_hectoliters",
    "weighted_distribution_tdp_reach", "year", "month", "SKU_ID"
] + ihs_vars

target = 'sales_hectoliters'

train_df = ce_df2[ce_df2['year_month'] < "2024-11-01"]
train_df = ce_df[ce_df['year_month'] < "2024-11-01"]
train_df['SKU_ID'] = train_df['SKU_ID'].astype('category')
train_df['year'] = train_df['year_month'].dt.year
train_df['month'] = train_df['year_month'].dt.month

X = train_df[features]
y = train_df[target]

X = X.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))

feature_names = list(X.columns)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(100)
]

best_params = study.best_params
best_params.update({
    "objective": "regression",
    "metric": "rmse",
    "boosting_type": "gbdt",
    "n_estimators": 5000,
    "random_state": 42
})

model_ce = lgb.LGBMRegressor(**best_params)
#model_ce.fit(X, y, categorical_feature=["SKU_ID"])
model_ce.fit(
    X, y,
    eval_set=[(X, y)],  
    eval_metric="rmse",
    callbacks = callbacks,
    #verbose=-1,
    categorical_feature=["SKU_ID"],
    feature_name=feature_names
)

# Predictions
y_pred_train_ce = model_ce.predict(X, num_iteration=model_ce.best_iteration_)
print("RMSE Train:", mean_squared_error(y, y_pred_train_ce, squared=False))
print("R² Train:", r2_score(y, y_pred_train_ce))
print('\n')

In [ ]:
full_df_test_sub = full_df_test[full_df_test['SKU_ID'].isin(full_df['SKU_ID'].unique())]
pred_df  = full_df_test_sub.copy()

pred_df["brand_avg_price"] = pred_df.groupby(["brand", "year_month"])["avg_price_per_liter"].transform("mean")
pred_df["subbrand_avg_price"] = pred_df.groupby(["sub_brand", "year_month"])["avg_price_per_liter"].transform("mean")
pred_df["package_avg_price"] = pred_df.groupby(["package_type", "year_month"])["avg_price_per_liter"].transform("mean")

pred_df['SKU_ID'] = pred_df['SKU_ID'].astype('category')
pred_df['year'] = pred_df['year_month'].dt.year
pred_df['month'] = pred_df['year_month'].dt.month

pred_df = pred_df.rename(columns=lambda x: str(x).replace(" ", "_").replace(",", "_").replace("(", "").replace(")", ""))
X_future = pred_df[X.columns]

# pred_df["log_volume_pred"] = model_ad.predict(X_future)

# pred_df["pred_volume"] = np.expm1(pred_df["log_volume_pred"])

pred_df["volume_pred"] = model_ce.predict(X_future, num_iteration=model_ce.best_iteration_)

pred_df.head(10)

In [ ]:
#pred_df.to_excel('lightgbm test data predictions OPTUNA AVERAGES.xlsx',index=False)